## Demo: Content-Based Recommender C (with metadata and review sentiment analysis)

This notebook implements the Content-Based C filtering recommender system (with metadata and review text) and applies it to a real user profile.

### Loading Data

Loads the three CSV files:

- restaurants_santa_barbara.csv
- test_reviews_santa_barbara.csv
- train_reviews_santa_barbara.csv

In [16]:
import pandas as pd
import numpy as np

#resturants
df_rests = pd.read_csv(
    "restaurants_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_rests.columns = df_rests.iloc[0]
df_rests = df_rests[1:]
df_rests = df_rests.reset_index(drop=True)

#test
df_test = pd.read_csv(
    "test_reviews_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_test.columns = df_test.iloc[0]
df_test = df_test[1:]
df_test = df_test.reset_index(drop=True)

#train
df_train = pd.read_csv(
    "train_reviews_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_train.columns = df_train.iloc[0]
df_train = df_train[1:]
df_train = df_train.reset_index(drop=True)

### Encode restaurant categories (multi-hot encodings)

Each restaurant has a "categories" field with a list of tags

Use multi-hot encoding: each unique category across all restaurants becomes one binary column. A restaurant gets a 1 in each column corresponding to its categories and 0 everywhere else.

This is better than a transformer embedding for categories because:
- Categories are discrete labels, not semantic text
- A transformer would produce 384 dense dimensions, drowning out the smaller attribute features
- Keeps each category as an interpretable, equally-weighted binary signal

In [17]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
# Multi-hot encode restaurant categories using get_dummies

# Split comma-separated categories string into a list per restaurant
df_rests["category_list"] = df_rests["categories"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",")]
)

# get_dummies for binary encoding
category_dummies = pd.get_dummies(
    df_rests["category_list"].explode()
).groupby(level=0).max()
category_dummies = category_dummies.reindex(df_rests.index, fill_value=0)


### Encode attribute features

Chose three attributes from the restaurant metadata:

- GoodForKids (binary): users with families will prefer family-friendly restaurants
- RestaurantsAttire (binary): users who prefer comfort seek casual restaurants
- RestaurantsPriceRange2 (multi-class: 1–4): price is an important determining factor for many customers

Encoding strategy:
- Binary attributes → 0 or 1 directly
- Multi-class price range → one-hot encoding with get_dummies, so each price level gets its own column

In [18]:
#on top of the categorical embeddings, I want to add to each embedding a few values connoting attribute values

#first clean values for the attribute column
import ast
from sklearn.preprocessing import normalize
from scipy.sparse import hstack, csr_matrix

def str_to_dict(s):
    """
    Safely parse a string representation of a Python dictionary.
    Args:
        s (str): A string that may contain a dictionary literal.
    Returns:
        dict: The parsed dictionary, or an empty dict if parsing fails.
    """
    try:
        return ast.literal_eval(s)
    except: #need this or else it won't run
        return {}
df_rests['attributes_dict'] = df_rests['attributes'].apply(str_to_dict)

def clean_values(d):
    """
    Strip extra quote characters from string values in a dictionary.
    Args:
        d (dict): A dictionary whose values may contain leading/trailing quotes.
    Returns:
        dict: The same dictionary with string values without quote characters.
    """
    return {k: (v.strip("'\"") if isinstance(v, str) else v) for k, v in d.items()}
df_rests['attributes_dict'] = df_rests['attributes_dict'].apply(clean_values)

# use RestaurantsPriceLevel2, GoodForKids, and RestaurantsAttire; these are all good indicators because people feel strongly about prices, about their families, and about "comfortability" of a restaurant. A person who wants to eat cheap will really appreciate a cheap recommendation, a person who has kids will really appreciate a restaurant that caters to them, and a person who likes to be comfortable/homey will really appreciate a restaurant recommendation that feels like home.

# Binary attribute: GoodForKids
df_rests["good_for_kids"] = df_rests["attributes_dict"].apply(
    lambda d: 1 if d.get("GoodForKids") == "True" else 0
)

# Binary attribute: RestaurantsAttire (casual vs not)
df_rests["casual"] = df_rests["attributes_dict"].apply(
    lambda d: 1 if d.get("RestaurantsAttire") == "casual" else 0
)

# Multi-class attribute: PriceRange — one-hot encode with get_dummies
# Restaurants with no price data are assigned "unknown" category
df_rests["price_range"] = df_rests["attributes_dict"].apply(
    lambda d: d.get("RestaurantsPriceRange2", "unknown")
)
price_dummies = pd.get_dummies(df_rests["price_range"], prefix="price")
price_dummies = price_dummies.reindex(df_rests.index, fill_value=0)

### Make weighted metadata vector

Three feature blocks (categories, price range, binary attributes) are combined into a single metadata vector with weighted concatenation. Categories are 0.6, price range are 0.25, and binary attributes are 0.15.

This ensures that the largest block (the categories) won't dominate the cosine similarity calculation just because of its larger dimensionality.

Combined vector is L2-normalized prior to computing cosine similarity.

In [19]:
X_cat = csr_matrix(category_dummies.values.astype(float))
X_attr = csr_matrix(price_dummies.values.astype(float))
X_binary = csr_matrix(df_rests[["good_for_kids", "casual"]].values.astype(float))

# set weights
weight_cat = 0.6
weight_attr = 0.25
weight_binary = 0.15

# Concatenate weighted blocks into a single matrix
X_meta_sparse = hstack([X_cat * weight_cat, X_attr * weight_attr, X_binary * weight_binary])
# L2 normalize each row so all metadata vectors have unit length
X_meta = normalize(X_meta_sparse, norm="l2", axis=1).toarray()

df_rests["meta_embedding"] = list(X_meta)
print("Metadata vector size:", X_meta.shape[1])

Metadata vector size: 232


### Aggregate reviews and embed

All training reviews for each restaurant are concatenated, then embedded using the all-MiniLM-L6-v2 sentence transformer

Aggregate reviews before embedding to produce one vector per restaurant that captures the overall language of its reviews. Restaurants with no training reviews will receive a zero vector.

Only training reviews are used. Test reviews are never seen during modeling.

In [20]:
from sentence_transformers import SentenceTransformer

embedder_model = SentenceTransformer("all-MiniLM-L6-v2")

# Aggregate all review texts per restaurant from training data
review_text_by_restaurant = df_train.groupby("business_id")["text"].apply(
    lambda texts: " ".join(texts.dropna())
).reset_index()
review_text_by_restaurant.columns = ["business_id", "aggregated_text"]

# Embed the aggregated review texts
review_embeddings = embedder_model.encode(review_text_by_restaurant["aggregated_text"].tolist())
review_text_by_restaurant["review_embedding"] = list(review_embeddings)

# Merge into restaurant dataframe
df_rests_b = df_rests.merge(
    review_text_by_restaurant[["business_id", "review_embedding"]],
    on="business_id",
    how="left"
)

# Fill restaurants with no reviews with a zero vector (same size as the model output)
embedding_size = review_embeddings.shape[1]  # 384 for all-MiniLM-L6-v2
df_rests_b["review_embedding"] = df_rests_b["review_embedding"].apply(
    lambda x: x if isinstance(x, np.ndarray) else np.zeros(embedding_size)
)

/Users/troyhealey/PycharmProjects/RecomSystemsClass/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Extract sentiment from reviews

For each restaurant, compute the sentiment score from its aggregated review text using the distilbert model.

Meaning of the score: positive values indicate overall positive sentiment, negative values
indicate negative sentiment, with magnitude reflecting confidence. The range is [-1, 1].

In [21]:
from transformers import pipeline

sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

def get_sentiment_score(text):
    """
    Compute a signed sentiment score for a piece of text using DistilBERT.

    The model classifies text as POSITIVE or NEGATIVE with a confidence score.
    This function returns the confidence score as positive for POSITIVE sentiment
    and negative for NEGATIVE sentiment.

    Args:
        text (str): The input text to analyze. Only the first 512 characters are used
            due to the model's token limit.

    Returns:
        float: A sentiment score in the range [-1, 1], where -1 is maximally negative
            and 1 is maximally positive.
    """
    result = sentiment_analyzer(text[:512])[0]
    score = result['score']
    if result['label'] == 'NEGATIVE':
        score = -score
    return score

# Apply sentiment scoring to each restaurant's aggregated review text
review_text_by_restaurant["sentiment_score"] = review_text_by_restaurant["aggregated_text"].apply(get_sentiment_score)

/Users/troyhealey/PycharmProjects/RecomSystemsClass/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Add sentiment score to embedding

### Make final feature vector

Create the final restaurant vector by horizontally concatenating the normalized metadata vector (categories, price range, binary attributes), the review embedding, and the sentiment score

The concatenated vector is L2 normalized again to ensure correct calculation of cosine similarity.

Final vector composition:
- Metadata block: weighted and normalized (size = num_categories + num_price_levels + 2)
- Review embedding block: 384 dimensions
- Sentiment score: 1 dimension
- Total: metadata_size + 384 + 1

In [22]:
# Merge sentiment scores into the restaurant dataframe
df_rests_c = df_rests_b.merge(
    review_text_by_restaurant[["business_id", "sentiment_score"]],
    on="business_id",
    how="left"
)
# Restaurants with no reviews get a neutral sentiment score of 0
df_rests_c["sentiment_score"] = df_rests_c["sentiment_score"].fillna(0.0)

X_meta = np.vstack(df_rests_c["meta_embedding"].values)
review_matrix = np.vstack(df_rests_c["review_embedding"].values)
sentiment_col = df_rests_c["sentiment_score"].values.reshape(-1, 1)

# Concatenate and L2 normalize the full vector
X_final = np.hstack([X_meta, review_matrix, sentiment_col])
X_final = normalize(X_final, norm="l2", axis=1)

df_rests_c["new_embedding"] = list(X_final)
print("Final vector size (C):", X_final.shape[1])
# Should be: metadata_size + 384 + 1

Final vector size (C): 617


### Build user profiles

Each user is represented by a profile vector computed as the mean of the feature vectors of restaurants they interacted with in the training set.


In [23]:
df_train['stars'] = pd.to_numeric(df_train['stars'])

# Compute per-user mean rating and adjusted rating
user_means = df_train.groupby("user_id")["stars"].mean()
df_train = df_train.merge(user_means.rename("user_mean"), on="user_id")
df_train["adjusted_rating"] = df_train["stars"] - df_train["user_mean"]

# Join training interactions with restaurant embeddings
merge = df_train.merge(df_rests_c, on="business_id")
merge = merge.set_index("business_id")

# Build user profiles as the mean embedding of all restaurants the user interacted with
user_profiles = {}
for user, group in df_train.groupby("user_id"):
    vectors = merge.loc[group["business_id"], "new_embedding"]
    user_profiles[user] = np.vstack(vectors).mean(axis=0)

### Picking a real user

Choosing a real user and printing out the information for all the restaurants that user has seen before.

In [24]:
# Picking a user and examining their training history
demo_user_id = "-1-ECBsGpG4Iw5s-ecnfqw"

user_history = df_train[df_train["user_id"] == demo_user_id].merge(
    df_rests_c[["business_id", "name", "categories", "price_range", "good_for_kids", "casual", "sentiment_score"]], on="business_id"
)

print(f"Restaurants this user has visited ({len(user_history)}):\n")
for _, row in user_history.iterrows():
    print(f"{row['name']}")
    print(f"    Categories: {row['categories']}")
    print(f"    Price Range: {row['price_range']}")
    print(f"    Good for Kids: {'Yes' if row['good_for_kids'] == 1 else 'No'}")
    print(f"    Attire: {'Casual' if row['casual'] == 1 else 'Not Casual'}")
    print(f"    Sentiment Score: {row['sentiment_score']:.3f}")
    print(f"    Stars: {row['stars']}\n")

print(f"Mean rating: {user_history['stars'].astype(float).mean():.2f}")

Restaurants this user has visited (8):

Loquita
    Categories: event planning & services, nightlife, tapas bars, spanish, modern european, bars, cocktail bars, seafood, venues & event spaces, restaurants
    Price Range: 3
    Good for Kids: No
    Attire: Not Casual
    Sentiment Score: 1.000
    Stars: 5.0

Mesa Verde
    Categories: garage door services, restaurants, live/raw food, food, home services, mediterranean, keys & locksmiths, burgers, vegetarian, juice bars & smoothies, vegan, salad
    Price Range: 2
    Good for Kids: Yes
    Attire: Not Casual
    Sentiment Score: 1.000
    Stars: 3.0

Lucky Penny
    Categories: american (new), breakfast & brunch, restaurants, salad, italian, nightlife, bars, beer, wine & spirits, sandwiches, coffee & tea, wine bars, food, pizza
    Price Range: 2
    Good for Kids: Yes
    Attire: Not Casual
    Sentiment Score: 0.979
    Stars: 5.0

Tyger Tyger
    Categories: restaurants, asian fusion
    Price Range: 2
    Good for Kids: No
    At

### User Profile Summary

This user is a high-rater (mean rating of 4.75 stars) who gravitates toward upscale-casual American dining and nightlife venues (bars, cocktail bars, and restaurants with drinks). They visit both price level 2 and 3 establishments and don't show a strong preference for family-friendly venues (only half their visited restaurants are kid-friendly). All the visited restaurants have very high sentiment scores (0.979–1.000), indicating they tend to frequent well-regarded places.


### Recommendation function

Given a user, compute cosine similarity between their profile vector and all candidate restaurant vectors, then return the top-k most similar restaurants.

Candidate restaurants are all restaurants that have not been seen yet by the user

In [25]:
#get recommendations
from sklearn.metrics.pairwise import cosine_similarity

def recommend_restaurants(user_id, user_embeddings, restaurants_df, k, seen_business_ids=None):
    """
    Generate top-k restaurant recommendations for a given user using cosine similarity.
    Args:
        user_id (str): The ID of the user to generate recommendations for.
        user_embeddings (dict): Mapping from user_id to their profile vector (np.ndarray).
        restaurants_df (pd.DataFrame): DataFrame containing restaurant feature vectors in the 'new_embedding' column.
        k (int): Number of top recommendations to return.
        seen_business_ids (set, optional): Set of business IDs the user has already interacted
        with in training. These will be excluded from recommendations. Defaults to None.

    Returns:
        pd.DataFrame: A DataFrame with columns ['business_id', 'name', 'similarity'], sorted by descending similarity score.
    """
    if seen_business_ids:
        restaurants_df = restaurants_df[~restaurants_df["business_id"].isin(seen_business_ids)].reset_index(drop=True)

    user_vec = user_embeddings[user_id].reshape(1, -1)
    restaurant_matrix = np.vstack(restaurants_df['new_embedding'].values)

    similarities = cosine_similarity(user_vec, restaurant_matrix)[0]
    top_k_idx = np.argsort(similarities)[-k:][::-1]

    results = restaurants_df[['business_id', 'name']].iloc[top_k_idx].copy()
    results['similarity'] = similarities[top_k_idx]

    return results

### Get recommendations for User

Getting recommendations for restaurants the user has not visited before. Prints out the informationn for each recommended restaurant.

In [26]:
# Only generate recommendations from list of unseen restaurants
seen = set(df_train[df_train["user_id"] == demo_user_id]["business_id"])
recommendations = recommend_restaurants(demo_user_id, user_profiles, df_rests_c, k=10, seen_business_ids=seen)

print("Top 10 recommendations:")
print(recommendations[["business_id", "name", "similarity"]].to_string(index=False))

recs_with_info = recommendations.merge(
    df_rests_c[["business_id", "categories", "price_range", "good_for_kids", "casual", "sentiment_score"]], on="business_id"
)

print(f"Top recommendations for user ({len(recs_with_info)}):\n")
for _, row in recs_with_info.iterrows():
    print(f"{row['name']}")
    print(f"    Categories: {row['categories']}")
    print(f"    Price Range: {row['price_range']}")
    print(f"    Good for Kids: {'Yes' if row['good_for_kids'] == 1 else 'No'}")
    print(f"    Attire: {'Casual' if row['casual'] == 1 else 'Not Casual'}")
    print(f"    Sentiment Score: {row['sentiment_score']:.3f}")
    print(f"    Similarity: {row['similarity']:.4f}\n")

Top 10 recommendations:
           business_id                            name  similarity
3Wy21heeDm8h2tSZfcj6OA                 Lure Fish House    0.843749
BseHdXex5Od2Or_TyUIoWQ               Courthouse Tavern    0.841477
CoZ2mpsMBP8HUG1ymKoZTg           Opal Restaurant & Bar    0.838501
jfRq1KtjEGyPoMo9HZVw2Q                         The Set    0.835646
zbrIMldF_O1ZQ0vpUaaa8A Bluewater Grill - Santa Barbara    0.835160
MOowZBGbgn7FrTol3dvrlg          Nectar Eatery & Lounge    0.831514
IRyKDIF51M0GcqS8yfPcYA                       Mesa Cafe    0.830313
XYxSB4DQ48O7J6AsnS-Ddg        State Street Wine Bistro    0.827591
pKSB4wXLiS8rd1cAEgD8oA                     Shaker Mill    0.826277
fiQxkb1pfPpvrr0oYYYBcA            The Daisy Restaurant    0.825567
Top recommendations for user (10):

Lure Fish House
    Categories: breakfast & brunch, food, beer, wine & spirits, bars, seafood, restaurants, wine bars, cocktail bars, american (new), nightlife
    Price Range: 2
    Good for Kids: Yes
 

### Recommendations Analysis

Strong category alignment. Nearly every recommendation falls into American (new/traditional), bars, nightlife, and seafood and these are the categories this user often visits. Lure Fish House, Courthouse Tavern, Opal Restaurant & Bar, and Bluewater Grill all match the user's dining profile closely.

Price range is consistent. 8 out of 10 recommendations are price level 2, with State Street Wine Bistro at level 3 — consistent with the user's training history which is mostly level 2 with some level 3.

Sentiment scores are all high. All recommended restaurants score between 0.996 and 1.000, matching the sentiment profile of the restaurants this user has enjoyed. Recommends places with overwhelmingly positive reviews.

Notes:
- Mesa Verde is an outlier since the user gave Mesa Verde only 3 stars, which was their only below-average rating. Mesa Verde has an unusual category mix ("garage door services", "keys & locksmiths") which suggests noisy metadata. The recommender correctly doesn't recommend anything similar to Mesa Verde.
- Shaker Mill is the only recommendation with unknown price range. The model handles missing price data by assigning it to the "unknown" one-hot column, so it still gets recommended based on its similarity for other features (strong category and review embedding match).
